# 正态性检验完整教程：从直方图到 Shapiro-Wilk

## 📚 教学目标
1. 掌握 Shapiro-Wilk 检验的使用
2. 学会用 QQ 图评估正态性
3. 理解偏度和峰度的含义
4. 综合多种方法判断数据是否正态
5. 理解正态性检验对后续统计推断的影响

**参考书**：《基础统计学(第14版)》（Triola）第 6-5 节
**教学策略**：用真实分布和模拟数据，对比正态与非正态数据的检验结果

## 1. 场景设定：为什么需要检验正态性？

### 🎯 问题
后续章节的许多统计方法（t 检验、方差分析、置信区间）都假设数据来自正态分布的总体。如果这个假设不成立，结论可能不可靠。

所以，在使用这些方法之前，我们需要先检验：**数据是否来自正态分布？**

### 📖 书中的方法
> 确定样本数据来自正态分布的总体是否合理的过程：
> 1. 构建直方图，检查是否大致为钟形曲线
> 2. 构建正态分位图（QQ 图），检查点是否接近直线
> 3. 使用高阶检验（如 Shapiro-Wilk 检验）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print("✅ 库导入完成")

## 2. 生成四种不同分布的样本数据

我们将用四种不同分布的数据来演示正态性检验：

| 数据集 | 分布 | 预期结果 |
|--------|------|----------|
| IQ 分数 | 正态分布 | 应该通过正态性检验 |
| 降雨量 | 右偏分布 | 应该不通过 |
| 均匀分布 | 平坦分布 | 应该不通过 |
| 指数分布 | 严重右偏 | 应该不通过 |

In [ ]:
# ========== 步骤 1: 生成四种样本数据 ==========

n_samples = 200

# 正态分布：IQ 分数
iq_scores = np.random.normal(100, 15, n_samples)

# 右偏分布：降雨量（指数分布 + 偏移）
rainfall = np.random.exponential(0.5, n_samples)

# 均匀分布
uniform_data = np.random.uniform(0, 10, n_samples)

# 指数分布
exponential_data = np.random.exponential(2, n_samples)

datasets = {
    'IQ Scores (Normal)': iq_scores,
    'Rainfall (Right-skewed)': rainfall,
    'Uniform Distribution': uniform_data,
    'Exponential Distribution': exponential_data
}

print(f"\n📊 步骤 1: 四种样本数据概况")
print(f"  样本量: {n_samples} 每组")
print(f"\n{'数据集':<25} {'均值':>8} {'标准差':>8} {'偏度':>8} {'峰度':>8}")
print("-" * 60)
for name, data in datasets.items():
    print(f"{name:<25} {np.mean(data):>8.2f} {np.std(data, ddof=1):>8.2f} "
          f"{stats.skew(data):>8.3f} {stats.kurtosis(data):>8.3f}")

In [ ]:
# ========== 可视化：四种数据的直方图 ==========

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, data) in zip(axes, datasets.items()):
    ax.hist(data, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='white')
    
    # 叠加正态曲线
    x = np.linspace(data.min(), data.max(), 100)
    y = stats.norm.pdf(x, np.mean(data), np.std(data, ddof=1))
    ax.plot(x, y, 'r-', linewidth=2, label='Normal Curve')
    
    ax.set_xlabel('Value', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Histograms: Visual Check for Normality', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  IQ 分数：接近钟形，与正态曲线吻合")
print(f"  降雨量：右偏，高峰在左侧")
print(f"  均匀分布：平坦，不是钟形")
print(f"  指数分布：严重右偏")

## 3. 方法一：直方图判断

### 📐 直方图的判断标准
- **正态**: 直方图大致呈钟形，左右对称
- **非正态**: 明显偏态、多峰、平坦等

### 💡 局限性
直方图是主观判断，不同的人可能得出不同结论。样本量小时尤其不可靠。

## 4. 方法二：QQ 图（正态分位图）

### 📐 QQ 图的原理
正态分位图（Normal Quantile Plot / QQ Plot）是一个散点图：
- **横坐标**: 原始样本值（从小到大排序）
- **纵坐标**: 标准正态分布对应的 z 分数

### 📖 书中的判断标准
> - **正态分布**: 如果图上点的分布接近一条直线，且没有非直线的系统性特征
> - **非正态分布**: 如果点有明显的曲线模式，或者有偏离直线的系统性特征

### 📐 QQ 图的手动构建步骤
1. 将数据从低到高排序
2. 计算每个值对应的累积面积：$\frac{i}{n}$（或更精确的 $\frac{i-0.5}{n}$）
3. 查找累积面积对应的 z 分数
4. 绘制 (原始值, z 分数) 的散点图

In [ ]:
# ========== 步骤 2: 手动构建 QQ 图（以 IQ 为例） ==========

# 取前 10 个 IQ 分数演示手动过程
iq_small = np.sort(iq_scores[:10])
n_small = len(iq_small)

# 计算累积面积
cum_areas = (np.arange(1, n_small + 1) - 0.5) / n_small

# 查找对应的 z 分数
z_scores = stats.norm.ppf(cum_areas)

print(f"\n📊 步骤 2: 手动构建 QQ 图（前 10 个 IQ 分数）")
print(f"{'排序后的值':>12} {'累积面积':>10} {'对应 z 分数':>12}")
print("-" * 38)
for val, area, z in zip(iq_small, cum_areas, z_scores):
    print(f"{val:>12.2f} {area:>10.3f} {z:>12.3f}")

print(f"\n💡 如果数据来自正态分布，(值, z 分数) 的散点应该接近一条直线")

In [ ]:
# ========== 步骤 3: 用 scipy 构建 QQ 图 ==========

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, data) in zip(axes, datasets.items()):
    # scipy 的 probplot 自动构建 QQ 图
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    
    ax.scatter(osm, osr, alpha=0.6, color='steelblue', s=30)
    
    # 理论直线
    x_line = np.array([min(osm), max(osm)])
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'R²={r:.4f}')
    
    ax.set_xlabel('Theoretical Quantiles', fontsize=10)
    ax.set_ylabel('Sample Quantiles', fontsize=10)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('QQ Plots: Visual Assessment of Normality', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  IQ 分数：点紧密围绕直线 → 正态分布 ✅")
print(f"  降雨量：点呈曲线偏离 → 右偏分布 ❌")
print(f"  均匀分布：两端偏离直线 → 非正态 ❌")
print(f"  指数分布：严重偏离直线 → 右偏分布 ❌")
print(f"  R² 越接近 1，数据越接近正态分布")

## 5. 方法三：Shapiro-Wilk 检验

### 📐 检验假设
- **H₀**: 数据来自正态分布
- **H₁**: 数据不来自正态分布

### 📐 判断标准
- 如果 **p > 0.05**，不拒绝 H₀，认为数据近似正态
- 如果 **p ≤ 0.05**，拒绝 H₀，认为数据不是正态分布

### 💡 注意
Shapiro-Wilk 检验对大样本非常敏感——即使轻微偏离正态也会被检测出来。因此，实际应用中要结合 QQ 图和偏度/峰度综合判断。

In [ ]:
# ========== 步骤 4: Shapiro-Wilk 检验 ==========

print(f"\n📊 步骤 4: Shapiro-Wilk 正态性检验")
print(f"  H₀: 数据来自正态分布")
print(f"  H₁: 数据不来自正态分布")
print(f"  显著性水平 α = 0.05")
print(f"\n{'数据集':<25} {'W 统计量':>12} {'p 值':>12} {'结论':>15}")
print("-" * 68)

for name, data in datasets.items():
    w_stat, p_val = stats.shapiro(data)
    conclusion = '✅ 正态' if p_val > 0.05 else '❌ 非正态'
    print(f"{name:<25} {w_stat:>12.4f} {p_val:>12.6f} {conclusion:>15}")

print(f"\n💡 解读：")
print(f"  W 统计量越接近 1，数据越接近正态分布")
print(f"  p > 0.05 → 不拒绝正态性假设")
print(f"  p ≤ 0.05 → 拒绝正态性假设")

## 6. 偏度和峰度

### 📐 偏度 (Skewness)
衡量分布的**不对称程度**：
- 偏度 = 0：完全对称
- 偏度 > 0：右偏（长尾在右）
- 偏度 < 0：左偏（长尾在左）

### 📐 峰度 (Kurtosis)
衡量分布的**尾部厚度**和**尖锐程度**：
- 峰度 = 0（超额峰度）：与正态分布相同
- 峰度 > 0：尖峰厚尾（leptokurtic）
- 峰度 < 0：平峰薄尾（platykurtic）

### 📐 判断标准
- |偏度| < 1 且 |峰度| < 1：可以认为近似正态
- |偏度| > 2 或 |峰度| > 7：明显非正态

In [ ]:
# ========== 步骤 5: 偏度和峰度分析 ==========

print(f"\n📊 步骤 5: 偏度和峰度分析")
print(f"\n{'数据集':<25} {'偏度':>10} {'|偏度|':>8} {'峰度':>10} {'|峰度|':>8} {'正态？':>10}")
print("-" * 75)

for name, data in datasets.items():
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)  # 超额峰度（正态分布为 0）
    is_normal = '✅' if abs(skew) < 1 and abs(kurt) < 1 else '❌'
    print(f"{name:<25} {skew:>10.3f} {abs(skew):>8.3f} {kurt:>10.3f} {abs(kurt):>8.3f} {is_normal:>10}")

print(f"\n💡 判断标准：")
print(f"  |偏度| < 1 且 |峰度| < 1 → 可认为近似正态")
print(f"  |偏度| > 2 或 |峰度| > 7 → 明显非正态")

In [ ]:
# ========== 可视化：偏度和峰度 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(datasets.keys())
skewness_vals = [stats.skew(d) for d in datasets.values()]
kurtosis_vals = [stats.kurtosis(d) for d in datasets.values()]

# 偏度
ax1 = axes[0]
colors = ['#2ecc71' if abs(s) < 1 else '#e74c3c' for s in skewness_vals]
bars1 = ax1.bar(range(len(names)), skewness_vals, color=colors, edgecolor='white', alpha=0.8)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(y=-1, color='gray', linestyle='--', alpha=0.5)
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, fontsize=9)
ax1.set_ylabel('Skewness', fontsize=12)
ax1.set_title('Skewness of Different Distributions', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 峰度
ax2 = axes[1]
colors2 = ['#2ecc71' if abs(k) < 1 else '#e74c3c' for k in kurtosis_vals]
bars2 = ax2.bar(range(len(names)), kurtosis_vals, color=colors2, edgecolor='white', alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(y=-1, color='gray', linestyle='--', alpha=0.5)
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, fontsize=9)
ax2.set_ylabel('Excess Kurtosis', fontsize=12)
ax2.set_title('Kurtosis of Different Distributions', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色：|值| < 1，可认为近似正态")
print(f"  红色：|值| ≥ 1，可能非正态")
print(f"  ➡ 偏度和峰度是快速筛查正态性的有效工具")

## 7. 综合判断：多方法联合决策

### 📐 正态性检验的综合流程
1. **直方图**: 初步视觉判断
2. **QQ 图**: 精确视觉判断
3. **偏度/峰度**: 量化偏离程度
4. **Shapiro-Wilk 检验**: 统计检验

### 💡 实际建议
不要只依赖一种方法。当多种方法的结论一致时，可以更有信心地做出判断。

In [ ]:
# ========== 步骤 6: 综合正态性评估报告 ==========

print(f"\n{'='*80}")
print(f"📋 综合正态性评估报告")
print(f"{'='*80}")

for name, data in datasets.items():
    print(f"\n{'─'*80}")
    print(f"📊 数据集: {name}")
    print(f"{'─'*80}")
    
    # 基本统计量
    print(f"  样本量: {len(data)}")
    print(f"  均值: {np.mean(data):.4f}")
    print(f"  标准差: {np.std(data, ddof=1):.4f}")
    
    # 偏度和峰度
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)
    print(f"  偏度: {skew:.4f} ({'轻微' if abs(skew) < 0.5 else '中等' if abs(skew) < 1 else '严重'}{'右偏' if skew > 0 else '左偏' if skew < 0 else ''})")
    print(f"  峰度: {kurt:.4f} ({'正常' if abs(kurt) < 1 else '异常'})")
    
    # Shapiro-Wilk 检验
    w_stat, p_val = stats.shapiro(data)
    print(f"  Shapiro-Wilk: W={w_stat:.4f}, p={p_val:.6f}")
    
    # QQ 图 R²
    (_, _), (_, _, r) = stats.probplot(data, dist='norm')
    print(f"  QQ Plot R²: {r:.4f}")
    
    # 综合判断
    checks = [
        ('偏度', abs(skew) < 1),
        ('峰度', abs(kurt) < 1),
        ('Shapiro-Wilk', p_val > 0.05),
        ('QQ R²', r > 0.98)
    ]
    
    print(f"  \n  综合判断：")
    for check_name, passed in checks:
        status = '✅ 通过' if passed else '❌ 未通过'
        print(f"    {check_name:<15} {status}")
    
    n_passed = sum(1 for _, p in checks if p)
    if n_passed >= 3:
        print(f"  \n  🎯 结论: 数据近似正态分布（{n_passed}/4 项通过）")
    elif n_passed >= 2:
        print(f"  \n  ⚠️ 结论: 数据可能近似正态，但需谨慎（{n_passed}/4 项通过）")
    else:
        print(f"  \n  ❌ 结论: 数据不是正态分布（{n_passed}/4 项通过）")

print(f"\n{'='*80}")

## 8. 小样本的 QQ 图：手动构建

### 📖 书中的例题
> 达拉斯通勤时间的前 5 个数据：{20, 16, 25, 10, 30}。构建正态分位图判断是否来自正态总体。

这个例子展示了 QQ 图的手动构建过程，帮助理解 QQ 图的原理。

In [ ]:
# ========== 步骤 7: 手动构建 QQ 图（书中的例子） ==========

# 原始数据
commute_raw = np.array([20, 16, 25, 10, 30])
commute_sorted = np.sort(commute_raw)
n_commute = len(commute_sorted)

# 步骤 1: 排序完成 ✓
# 步骤 2: 计算累积面积
cum_areas_commute = np.array([0.1, 0.3, 0.5, 0.7, 0.9])  # i/(n+1) 的简化

# 步骤 3: 查找对应的 z 分数
z_commute = stats.norm.ppf(cum_areas_commute)

print(f"\n📊 步骤 7: 手动构建 QQ 图")
print(f"  原始数据: {commute_raw}")
print(f"  排序后: {commute_sorted}")
print(f"\n{'序号':>4} {'排序值':>8} {'累积面积':>10} {'z 分数':>10} {'(x, z) 坐标':>18}")
print("-" * 55)
for i, (val, area, z) in enumerate(zip(commute_sorted, cum_areas_commute, z_commute)):
    print(f"{i+1:>4} {val:>8.0f} {area:>10.1f} {z:>10.3f} ({val:>3.0f}, {z:>6.3f})")

# 绘制 QQ 图
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(z_commute, commute_sorted, color='steelblue', s=100, zorder=5)

# 拟合直线
slope, intercept = np.polyfit(z_commute, commute_sorted, 1)
x_line = np.array([min(z_commute) - 0.5, max(z_commute) + 0.5])
ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=2, label='Fitted Line')

# 标注每个点
for val, z in zip(commute_sorted, z_commute):
    ax.annotate(f'({val}, {z:.2f})', (z, val), textcoords="offset points", 
               xytext=(10, 5), fontsize=9)

ax.set_xlabel('Z Score (Theoretical Quantiles)', fontsize=12)
ax.set_ylabel('Commute Time (Sample Quantiles)', fontsize=12)
ax.set_title('QQ Plot: Dallas Commute Times (n=5)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  5 个点大致在一条直线上 → 可以认为来自正态分布")
print(f"  但样本量太小（n=5），结论不太可靠")
print(f"  ➡ 大样本时 QQ 图更可靠")

## 9. 达拉斯通勤时间：大样本正态性检验

### 📖 书中的例子
> 使用 1000 个达拉斯通勤时间数据进行正态性检验。直方图显示高峰偏左，QQ 图显示点完全偏离直线，得出结论：数据不是来自正态分布。

In [ ]:
# ========== 步骤 8: 模拟达拉斯通勤时间 ==========

# 模拟右偏的通勤时间数据（类似书中的分布）
commute_dallas = np.random.exponential(scale=25, size=1000)
commute_dallas = commute_dallas[commute_dallas < 150]  # 截断

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 直方图
ax1 = axes[0]
ax1.hist(commute_dallas, bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='white')
x_fit = np.linspace(0, commute_dallas.max(), 100)
y_fit = stats.norm.pdf(x_fit, np.mean(commute_dallas), np.std(commute_dallas, ddof=1))
ax1.plot(x_fit, y_fit, 'r-', linewidth=2, label='Normal Curve')
ax1.set_xlabel('Commute Time (min)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Histogram', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# QQ 图
ax2 = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(commute_dallas, dist='norm')
ax2.scatter(osm, osr, alpha=0.4, color='steelblue', s=15)
x_line = np.array([min(osm), max(osm)])
ax2.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'R²={r:.4f}')
ax2.set_xlabel('Theoretical Quantiles', fontsize=12)
ax2.set_ylabel('Sample Quantiles', fontsize=12)
ax2.set_title('QQ Plot', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# Shapiro-Wilk 检验结果
ax3 = axes[2]
w_stat, p_val = stats.shapiro(commute_dallas)
skew_val = stats.skew(commute_dallas)
kurt_val = stats.kurtosis(commute_dallas)

ax3.axis('off')
text = f"""Normality Assessment
─────────────────────
Sample Size: {len(commute_dallas)}
Mean: {np.mean(commute_dallas):.2f}
Std Dev: {np.std(commute_dallas, ddof=1):.2f}
Skewness: {skew_val:.4f}
Kurtosis: {kurt_val:.4f}
─────────────────────
Shapiro-Wilk:
  W = {w_stat:.4f}
  p = {p_val:.2e}
─────────────────────
QQ Plot R² = {r:.4f}
─────────────────────
Conclusion: NOT Normal
(Right-skewed distribution)"""
ax3.text(0.1, 0.5, text, transform=ax3.transAxes, fontsize=12,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Dallas Commute Times: Normality Assessment', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  直方图：高峰偏左，严重右偏，不是钟形")
print(f"  QQ 图：点严重偏离直线，呈曲线模式")
print(f"  Shapiro-Wilk: p ≈ 0，强烈拒绝正态性假设")
print(f"  ➡ 达拉斯通勤时间不是正态分布，后续分析需使用非参数方法！")

## 10. 数据转换：将非正态数据转为正态

### 📖 书中的建议
> 许多数据集不呈正态分布，但可以对数据进行转换以达到修改后的值具有正态分布的特征。一种常见的转换是对 x 取对数。

### 📐 常见转换方法
- **对数转换**: $y = \ln(x)$ 或 $y = \log_{10}(x)$
- **平方根转换**: $y = \sqrt{x}$
- **倒数转换**: $y = 1/x$
- **Box-Cox 转换**: 自动选择最佳转换参数

In [ ]:
# ========== 步骤 9: 对数转换示例 ==========

# 对右偏的通勤时间数据取对数
commute_log = np.log(commute_dallas)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 转换前
ax1 = axes[0]
ax1.hist(commute_dallas, bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='white')
w_before, p_before = stats.shapiro(commute_dallas)
ax1.set_xlabel('Commute Time (min)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title(f'Original Data (Shapiro p={p_before:.2e})', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)

# 转换后
ax2 = axes[1]
ax2.hist(commute_log, bins=40, density=True, alpha=0.6, color='#2ecc71', edgecolor='white')
x_norm = np.linspace(commute_log.min(), commute_log.max(), 100)
y_norm = stats.norm.pdf(x_norm, np.mean(commute_log), np.std(commute_log, ddof=1))
ax2.plot(x_norm, y_norm, 'r-', linewidth=2)
w_after, p_after = stats.shapiro(commute_log)
ax2.set_xlabel('Log(Commute Time)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title(f'Log-Transformed (Shapiro p={p_after:.4f})', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.suptitle('Effect of Log Transformation on Normality', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  转换前：严重右偏，Shapiro-Wilk p ≈ 0（非正态）")
print(f"  转换后：接近正态，Shapiro-Wilk p = {p_after:.4f}")
print(f"  ➡ 对数转换可以将右偏数据转为近似正态！")
print(f"  如果对数服从正态分布，则原始数据服从\"对数正态分布\"。")


## 📌 核心概念回顾

### 📌 直方图 (Histogram)
- **用途**: 初步判断数据是否呈钟形
- **优点**: 直观
- **缺点**: 主观，小样本不可靠

### 📌 QQ 图 (Normal Quantile Plot)
- **原理**: 将样本分位数与理论正态分位数对比
- **判断**: 点接近直线 → 正态；曲线偏离 → 非正态
- **R²**: 越接近 1 越正态

### 📌 Shapiro-Wilk 检验
- **H₀**: 数据来自正态分布
- **p > 0.05**: 不拒绝正态性
- **p ≤ 0.05**: 拒绝正态性
- **注意**: 大样本时过于敏感

### 📌 偏度和峰度
- **偏度**: 衡量不对称程度（正态 = 0）
- **峰度**: 衡量尾部厚度（正态 = 0）
- **判断**: |偏度| < 1 且 |峰度| < 1 → 近似正态

### 🔗 正态性检验流程
```
直方图 → QQ 图 → 偏度/峰度 → Shapiro-Wilk → 综合判断
  ↓        ↓        ↓            ↓            ↓
初步判断  精确判断  量化偏离    统计检验    最终结论
```

## ❌ 常见误区

### ❌ 误区 1: 只用 Shapiro-Wilk 检验就够了
**✓ 正确理解**: Shapiro-Wilk 对大样本非常敏感，即使轻微偏离也会被检测出来。应结合 QQ 图和偏度/峰度综合判断。

### ❌ 误区 2: p > 0.05 就是正态分布
**✓ 正确理解**: p > 0.05 只是说"没有足够证据拒绝正态性"，不是"证明了正态性"。小样本时检验功效不足，可能漏检。

### ❌ 误区 3: 非正态数据就不能做任何分析
**✓ 正确理解**: 非正态数据可以：(1) 做数据转换使其正态；(2) 使用非参数检验（如 Wilcoxon、Kruskal-Wallis）；(3) 大样本时 CLT 保证均值近似正态。

### ❌ 误区 4: QQ 图的 R² = 0.99 就一定是正态
**✓ 正确理解**: R² 高只是必要条件，不是充分条件。还要检查是否有系统性偏离模式（如 S 形曲线）。

### ❌ 误区 5: 正态性检验通过了，后续分析一定正确
**✓ 正确理解**: 正态性只是许多统计方法的假设之一。还需要检查独立性、方差齐性等其他假设。正态性检验通过不等于所有假设都满足。